In [2]:
FORCE_RELOAD = False
DATASET_NAME = 'DiscoveryWorld'
DATASET_VERSION = 'v2'
DELETE_IF_EXISTS_DATASET = True


In [3]:
from dataset_manager import datasetManager


df= datasetManager.load_dataset(FORCE_RELOAD)
df

Loading dataset: dataset.parquet


,id,scenarioName,difficultyStr,seed,numUserAgents,maxAgents,taskName,taskDescription,maxScore,scorecard
0,0,Tutorial,Normal,0,1,1,TutorialTask,Where am I?! You just wake up in a strange pla...,3,"[{'name': 'Talk to elder', 'description': 'The..."
1,1,Combinatorial Chemistry,Easy,0,1,1,RustedKeyTaskEasy,You were venturing into the wilderness of Plan...,12,"[{'name': 'Take key', 'description': 'The key ..."
2,2,Combinatorial Chemistry,Normal,0,1,1,RustedKeyTaskNormal,You were venturing into the wilderness of Plan...,12,"[{'name': 'Take key', 'description': 'The key ..."
3,3,Combinatorial Chemistry,Challenge,0,1,1,RustedKeyTaskChallenge,You were venturing into the wilderness of Plan...,12,"[{'name': 'Take key', 'description': 'The key ..."
4,4,Archaeology Dating,Easy,0,1,1,ArchaeologyDigTaskEasy,You are in archaeological dig store room on Pl...,6,"[{'name': 'Take radiocarbon meter', 'descripti..."
5,5,Archaeology Dating,Normal,0,1,5,ArchaeologyDigTaskNormal,You are on an archaeological dig on Planet X. ...,10,"[{'name': 'Take radiocarbon meter', 'descripti..."
6,6,Archaeology Dating,Challenge,0,1,5,ArchaeologyDigTaskGenericRadioisotope,You are on an archaeological dig on Planet X. ...,13,"[{'name': 'Take radioisotope meter', 'descript..."
7,7,Plant Nutrients,Easy,0,1,1,SoilNutrientTaskEasy,You are at a botanical research station on Pla...,4,"[{'name': 'Soil Nutrient Meter Present', 'desc..."
8,8,Plant Nutrients,Normal,0,1,5,SoilNutrientTaskNormal,You are at a botanical research station on Pla...,9,"[{'name': 'Soil Nutrient Meter Present', 'desc..."
9,9,Plant Nutrients,Challenge,0,1,5,SoilNutrientTaskChallenge,You are at a botanical research station on Pla...,9,"[{'name': 'Soil Nutrient Meter Present', 'desc..."


In [112]:
df = df[
    (df["scenarioName"] == "Reactor Lab")
    & (df["difficultyStr"] == "Easy")
]
df
# print(df.iloc[10]['taskDescription'])

# print(df.iloc[10]['taskDescription'])

,id,scenarioName,difficultyStr,seed,numUserAgents,maxAgents,taskName,taskDescription,maxScore,scorecard
10,10,Reactor Lab,Easy,0,1,3,ReactorTaskEasy,You are at the Quantum Reactor Lab on Planet X...,11,"[{'name': 'Quantum Crystals Taken', 'descripti..."


In [113]:
df.columns

Index(['id', 'scenarioName', 'difficultyStr', 'seed', 'numUserAgents',
       'maxAgents', 'taskName', 'taskDescription', 'maxScore', 'scorecard'],
      dtype='str')

In [114]:
from langsmith import Client
from langsmith.schemas import Dataset
from langsmith.utils import LangSmithConflictError
from pandas import DataFrame

client = Client()

def create_examples(dataset: Dataset, df: DataFrame,):
    # if PROTECT_EXPERIMENTS:
    #     print("PROTECT_EXPERIMENTS enabled, skipping...")
    #     return
    
    example_ids = []
    for _,row in df.iterrows():
        example = client.create_example(
            dataset_id=dataset.id,            
            inputs= {
                "id": row["id"],
                "scenarioName": row["scenarioName"],
                "difficultyStr": row["difficultyStr"],
                "maxAgents": row["maxAgents"],
                "taskName": row["taskName"],
                "taskDescription": row["taskDescription"],
                "scorecard": row["scorecard"]                
            },
            outputs= {
                "maxScore": row["maxScore"]                         
            },
            # metadata={
            #     "category": row["category"],
            #     "intent": row["intent"],
            # }
        )

        example_ids.append(example.id)

def create_dataset(name, description, delete_if_exists= False):
    try:        
        if client.has_dataset(dataset_name=name):
            if delete_if_exists:
                print(f"Dataset '{name}' already exists. Deleting existing dataset...")
                client.delete_dataset(dataset_name=name)
            else:
                print(f"Dataset '{name}' already exists. Skipping and retrieving from Langsmith...")
                dataset = client.read_dataset(dataset_name=name)
                examples = client.list_examples(dataset_id=dataset.id)
                examples = list(examples)
                return dataset, examples

        dataset = client.create_dataset(dataset_name = name, description = description)
        print(f"Dataset '{name}' created successfully.")
        
        print(f"Creating examples: {len(df)}")
        create_examples(dataset, df)
        
        examples = client.list_examples(dataset_id=dataset.id)
        examples = list(examples)
        return dataset, examples    
    except LangSmithConflictError as e:
        print(f"Error on creating Dataset '{name}': {str(e)}")  

dataset_name = f'{DATASET_NAME} - {DATASET_VERSION}'

dataset, examples = create_dataset(
    name = dataset_name,
    description=dataset_name,
    delete_if_exists= DELETE_IF_EXISTS_DATASET
)

Dataset 'DiscoveryWorld - v2' created successfully.
Creating examples: 1


In [115]:
from dataclasses import dataclass

from agent_runner import AgentRunner

@dataclass
class AgentConfiguration():
    agent_type: str
    model_name: str | None = None
    num_agents: int = 1    
        
def target_function(inputs:dict, agent_config: AgentConfiguration) -> dict:
    result = AgentRunner().runAgent( 
                           agent = agent_config.agent_type, 
                           scenarioName= inputs["scenarioName"],
                           difficultyStr=inputs["difficultyStr"],
                           numSteps=50,
                           )    
    return {
        "agent_result": result
    }
        
def agent_evaluator(inputs: dict, # what was passed into your target function (e.g., the question text).
                    outputs:dict, # what your target function returned (e.g., the model’s answer).
                    reference_outputs:dict # the ground truth answers you attached to each dataset example 
                    ) -> dict:
    result = outputs["agent_result"]
    print(f"agent_result: {result}") 
    return {
        "results": [
            {"key": "completedSuccessfully", "score": bool(result["completedSuccessfully"])},
            {"key": "totalScore", "score": result["totalScore"]},
            {"key": "finalNormalizedScore", "score": result["finalNormalizedScore"]},
            {"key": "stepsTaken", "score": result["stepsTaken"]},
            {"key": "actionsTaken", "score": result["actionsTaken"]},
            {"key": "actionsSucceeded", "score": result["actionsSucceeded"]},
            {"key": "choosingActionTime", "score": result["choosingActionTime"]},
            {"key": "totalTimeTaken", "score": result["totalTimeTaken"]},
            {"key": "stepsPerSecond", "score": result["stepsPerSecond"]},
        ]
    }
       
def evaluate_agent(agent_config: AgentConfiguration):
    # agent_model_name_sanitized = agent_config.model_name.replace("/", "-")
    
    result = client.evaluate(                                 
        lambda inputs:  target_function(inputs, agent_config),        
        data = examples,
        evaluators = [agent_evaluator],
        experiment_prefix = f"wd-{agent_config}",
        metadata={
            # "stage": stage,    
            "agent": agent_config.agent_type,       
            "model": agent_config.model_name,
            "num_agents": agent_config.num_agents,
            # "judge_model": judge_model_name,
            # "training_mode": training_mode,
            # "agentic_memory": agentic_memory_mode,
            # "split": split["name"],
            "n_examples": len(df)
        },                
    )
    return result


In [116]:
from agents.AgentsFactory import  AgentType
model_name= "gpt-6-luna"
str(AgentConfiguration(agent_type=AgentType.LLMAgent, model_name = model_name, num_agents=1))

"AgentConfiguration(agent_type=<AgentType.LLMAgent: 'LLMAgent'>, model_name='gpt-6-luna', num_agents=1)"

In [117]:
from agents.AgentsFactory import  AgentType
model_name= "gpt-6-luna"
 
agent_configurations =[
    AgentConfiguration(agent_type=AgentType.Random),
    AgentConfiguration(agent_type=AgentType.LLMAgent, model_name = model_name, num_agents=1),
    AgentConfiguration(agent_type=AgentType.LLMAgent, model_name = "gpt-6-astra", num_agents=1),
    AgentConfiguration(agent_type=AgentType.LLMAgent, model_name = model_name, num_agents=2),
]

def evaluation_pipeline(agent_configurations):
    results = []
    for config in agent_configurations:
        result = evaluate_agent(config)
        results.append(result)
        
results = evaluation_pipeline(agent_configurations)        
results

View the evaluation results for experiment: 'wd-AgentConfiguration(agent_type=<AgentType.Random: 'Random'>, model_name=None, num_agents=1)-bb5262cd' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/bd761c09-c4cf-4c88-bf37-708ea389165a/compare?selectedSessions=6474310f-c791-43d1-a567-1272ad9084d2


Running agent 'Random' on scenario 'Reactor Lab' with difficulty 'Easy' for 50 steps, seed 0, thread ID 1, agents 1.
Loading sprites from index file: C:\dev\AI\Dundalk\advanced_agentic\project_code\environment\discoveryworld\discoveryworld\assets\spriteIndex.json
	Loaded 36 sprites from pixymoon/CuteRPG_Interior/32x32/CuteRPG_Interior_custom.png.
	Loaded 22 sprites from pixymoon/CuteRPG_Houses/32x32/CuteRPG_Houses_A-modified1.png.
	Loaded 30 sprites from pixymoon/CuteRPG_Desert/32x32/CuteRPG_Desert_Outside_modified.png.
	Loaded 1 sprites from pixymoon/CuteRPG_Desert/32x32/CuteRPG_Desert_Inside.png.
	Loaded 9 sprites from pixymoon/CuteRPG_Village/32x32/CuteRPG